# System tray

> Provides system-tray controls for monitoring, reporting, pause and resume, and application exit.

This module connects the tray interface to the monitoring controller. It provides actions for opening last-session and current-session reports, pausing monitoring for fixed or custom durations, resuming monitoring, and shutting down the application.

The tray icon runs on a daemon thread while Tkinter’s event loop runs on the calling thread. Monitoring lifecycle and report-data generation remain the controller’s responsibility.

In [ ]:
#| default_exp tray

In [ ]:
#| hide
from nbdev.showdoc import *

## Runtime dependencies

In [ ]:
#| export
import threading
import tkinter as tk
from tkinter import simpledialog
from queue import Empty, Queue


In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import snooper_pkg.config as cf
from snooper_pkg.report_window import *

## Dispatcher

In [ ]:
#| export
def open_last_session_report(
    controller, # Controller used to generate last-session report data
    root,
    submit_ui
):
    """Show the last-session report when report data is available."""
    report_data = controller.generate_last_report_data()
    if report_data:
        submit_ui(
            show_report,
            root,
            report_data,
            "Last Session Report",
        )

def open_current_session_report(
    controller, # Controller used to generate current-session report data
    root,
    submit_ui
):
    """Show the current-session report when report data is available."""
    report_data = controller.generate_current_session_stats()
    if report_data:
        submit_ui(
            show_report,
            root,
            report_data,
            "Current Session Report",
        )

In [ ]:
#| export
class UiDispatcher:
    def __init__(self, root):
        self.root = root
        self.queue = Queue()
        self.root.after(100, self._process)

    def submit(self, callback, *args, **kwargs):
        self.queue.put((callback, args, kwargs))

    def _process(self):
        try:
            while True:
                callback, args, kwargs = self.queue.get_nowait()
                callback(*args, **kwargs)
        except Empty:
            pass
        finally:
            self.root.after(100, self._process)


Dispatches callbacks from background threads to Tkinter’s main thread, ensuring UI updates are executed safely through the root event loop.

## Report actions

## Tray menu and event loop

In [ ]:
#| export
def run_tray(
    controller, # Controller that owns monitoring and pause/resume state
    show_last_on_start=False # Whether to show the last session report on app start up
):
    """Start the tray integration and block in the Tkinter event loop until exit."""
    import pystray
    from PIL import Image

    root = tk.Tk()
    root.protocol("WM_DELETE_WINDOW", root.destroy)
    root.withdraw()  # Hide the main window

    ui = UiDispatcher(root)

    def close_tk():
        root.quit()
        root.destroy()
    def on_exit(icon, item):
        if controller.is_running(): controller.stop_monitoring(stop_reason="manual_stop")
        icon.stop()
        ui.submit(close_tk)
    
    def pause_for(icon, item, minutes):
        controller.pause_for(minutes)
        icon.update_menu()
        open_current_session_report(controller, root, ui.submit)
    
    def pause_custom(icon, item):
        # Prompt user for custom pause duration
        ui.submit(ask_pause_duration, icon)

    def ask_pause_duration(icon):
        minutes = simpledialog.askinteger("Pause Duration", "Enter pause duration in minutes:", minvalue=1)
        if minutes:
            controller.pause_for(minutes)
            icon.update_menu()
            open_current_session_report(controller, root, ui.submit)

    def resume_now(icon, item):
        controller.resume_now()
        icon.update_menu()
   
    menu = pystray.Menu(
            pystray.MenuItem("Show last session report",
                            lambda icon, item: open_last_session_report(
                                controller, root, ui.submit
                                )
                            ),
            pystray.MenuItem("Show current session stats", 
                            lambda icon, item: open_current_session_report(
                                controller, root, ui.submit
                            ),
                            enabled=lambda item: controller.is_running()
            ),
            pystray.MenuItem(
                "Pause 15 min", 
                lambda icon, item: pause_for(icon, item, 15),
                enabled=lambda item: controller.is_running()
            ),
            pystray.MenuItem(
                "Pause 30 min", 
                lambda icon, item: pause_for(icon, item, 30),
                enabled=lambda item: controller.is_running()
            ),
            pystray.MenuItem("Pause for...", pause_custom, 
                         enabled=lambda item: controller.is_running()
            ),
            pystray.MenuItem(
                "Resume now",
                lambda icon, item: resume_now(icon, item),
                enabled=lambda item: controller.is_paused()
            ),
            pystray.MenuItem("Exit", on_exit),
        )

    image = Image.new("RGB", (64, 64), "blue")
    icon = pystray.Icon("Snooper", image, "Snooper", menu)
    controller.on_state_change = lambda: icon.update_menu()
    controller.on_notify = lambda msg: icon.notify(msg, "Snooper")
    icon.run_detached()

    # Schedule startup report after the Tk event loop begins.
    if show_last_on_start:
        root.after(
            0,
            lambda: open_last_session_report(
                controller,
                root,
                ui.submit,
            ),
        )

    root.mainloop()

- `root = tk.Tk()` runs at module-import time. This creates a GUI object as an import side effect and will also fail in headless environments. Confirm that import-time Tk initialization is intentional.

- The report helpers call `show_report` from new daemon threads. This appears inconsistent with the project rule that Tkinter UI operations should run on the Tk main thread, preferably through `root.after(...)`.

- After pausing, `pause_for` calls `open_current_session_report(controller)`. Because pausing ends the current session, confirm whether this should instead display the report for the just-ended session.

- Cancelling the custom pause dialog performs no pause. It does not use `DEFAULT_PAUSE_MINUTES`; confirm whether this is the intended current behaviour.

- `root` is exported as part of the module’s public API because its cell uses `#| export`. Confirm whether it should remain public or eventually become an internal implementation detail.

- The public functions accept an untyped `controller`. The comments document how it is used, but the implementation does not identify a concrete controller type, so no type annotation has been invented.

- The current notebook output shows that the failed `pystray` import prevented `tk` from being imported, which caused the later `root = tk.Tk()` cell to fail with `NameError`. This is a cascading notebook-execution failure rather than an independent missing import.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()